In [1]:
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully")

Mounted at /content/drive
Google Drive mounted successfully


In [2]:
import pandas as pd

csv_path = (
    "/content/drive/MyDrive/"
    "Hyderabad_Urban_Heat_Analytics/"
    "data/processed/"
    "hyderabad_urban_heat_analytics_2025.csv"
)

df = pd.read_csv(csv_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df.columns.tolist())

Rows: 150
Columns: 22
['ward_id', 'ward_name', 'circle', 'zone', 'area_sq_km', 'population_2011', 'households_2011', 'population_density_2011', 'lst_2025_c', 'ndvi_2025', 'ndbi_2025', 'heat_percentile', 'vegetation_percentile', 'built_up_percentile', 'density_percentile', 'heat_rank', 'vegetation_rank', 'built_up_intensity_rank', 'heat_level', 'vegetation_level', 'built_up_level', 'heat_density_category']


In [3]:
print(df.dtypes)

ward_id                      int64
ward_name                   object
circle                      object
zone                        object
area_sq_km                 float64
population_2011            float64
households_2011            float64
population_density_2011    float64
lst_2025_c                 float64
ndvi_2025                  float64
ndbi_2025                  float64
heat_percentile            float64
vegetation_percentile      float64
built_up_percentile        float64
density_percentile         float64
heat_rank                    int64
vegetation_rank              int64
built_up_intensity_rank      int64
heat_level                  object
vegetation_level            object
built_up_level              object
heat_density_category       object
dtype: object


In [4]:
sql_csv_path = (
    "/content/drive/MyDrive/"
    "Hyderabad_Urban_Heat_Analytics/"
    "data/processed/"
    "hyderabad_urban_heat_analytics_2025_sql.csv"
)

df.to_csv(sql_csv_path, index=False)

print("SQL-ready CSV created successfully")
print(sql_csv_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))

SQL-ready CSV created successfully
/content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/data/processed/hyderabad_urban_heat_analytics_2025_sql.csv
Rows: 150
Columns: 22


In [5]:
import pandas as pd

sql_path = (
    "/content/drive/MyDrive/"
    "Hyderabad_Urban_Heat_Analytics/"
    "data/processed/"
    "ward_heat_analysis_insert.sql"
)

columns = df.columns.tolist()

def sql_value(value):
    if pd.isna(value):
        return "NULL"
    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"
    return str(value)

with open(sql_path, "w", encoding="utf-8") as f:
    f.write("USE hyderabad_urban_heat_analytics;\n\n")
    f.write(
        "INSERT INTO ward_heat_analysis ("
        + ", ".join(f"`{col}`" for col in columns)
        + ")\nVALUES\n"
    )

    rows = []
    for _, row in df.iterrows():
        values = ", ".join(sql_value(row[col]) for col in columns)
        rows.append(f"({values})")

    f.write(",\n".join(rows))
    f.write(";\n")

print("SQL INSERT file created successfully")
print(sql_path)
print("Rows prepared:", len(df))

SQL INSERT file created successfully
/content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/data/processed/ward_heat_analysis_insert.sql
Rows prepared: 150


In [6]:
import os

sql_dir = (
    "/content/drive/MyDrive/"
    "Hyderabad_Urban_Heat_Analytics/"
    "sql"
)

os.makedirs(sql_dir, exist_ok=True)

sql_analysis_path = os.path.join(
    sql_dir,
    "analysis_queries.sql"
)

sql_content = """USE hyderabad_urban_heat_analytics;

SELECT
    ward_id,
    ward_name,
    zone,
    circle,
    lst_2025_c,
    heat_level,
    heat_rank
FROM ward_heat_analysis
ORDER BY lst_2025_c DESC
LIMIT 10;

SELECT
    zone,
    COUNT(*) AS ward_count,
    ROUND(AVG(lst_2025_c), 2) AS avg_lst_c,
    ROUND(MIN(lst_2025_c), 2) AS min_lst_c,
    ROUND(MAX(lst_2025_c), 2) AS max_lst_c,
    ROUND(AVG(ndvi_2025), 4) AS avg_ndvi,
    ROUND(AVG(ndbi_2025), 4) AS avg_ndbi
FROM ward_heat_analysis
GROUP BY zone
ORDER BY avg_lst_c DESC;

SELECT
    ward_id,
    ward_name,
    zone,
    ndbi_2025,
    lst_2025_c,
    built_up_level,
    heat_level
FROM ward_heat_analysis
ORDER BY ndbi_2025 DESC
LIMIT 15;

SELECT
    built_up_level,
    COUNT(*) AS ward_count,
    ROUND(AVG(ndbi_2025), 4) AS avg_ndbi,
    ROUND(AVG(lst_2025_c), 2) AS avg_lst_c,
    ROUND(MIN(lst_2025_c), 2) AS min_lst_c,
    ROUND(MAX(lst_2025_c), 2) AS max_lst_c
FROM ward_heat_analysis
GROUP BY built_up_level
ORDER BY avg_lst_c DESC;

SELECT
    vegetation_level,
    COUNT(*) AS ward_count,
    ROUND(AVG(ndvi_2025), 4) AS avg_ndvi,
    ROUND(AVG(lst_2025_c), 2) AS avg_lst_c,
    ROUND(MIN(lst_2025_c), 2) AS min_lst_c,
    ROUND(MAX(lst_2025_c), 2) AS max_lst_c
FROM ward_heat_analysis
GROUP BY vegetation_level
ORDER BY avg_lst_c DESC;

SELECT
    ward_id,
    ward_name,
    zone,
    population_2011,
    population_density_2011,
    lst_2025_c,
    heat_level,
    heat_density_category
FROM ward_heat_analysis
WHERE heat_density_category = 'High Heat / High Density'
ORDER BY lst_2025_c DESC;

SELECT
    heat_density_category,
    COUNT(*) AS ward_count,
    ROUND(AVG(lst_2025_c), 2) AS avg_lst_c,
    ROUND(AVG(population_density_2011), 2) AS avg_density,
    ROUND(SUM(population_2011), 0) AS population_2011
FROM ward_heat_analysis
WHERE heat_density_category <> 'NA'
GROUP BY heat_density_category
ORDER BY avg_lst_c DESC;

SELECT
    heat_level,
    COUNT(*) AS ward_count,
    ROUND(AVG(lst_2025_c), 2) AS avg_lst_c,
    ROUND(AVG(population_density_2011), 2) AS avg_density,
    ROUND(SUM(population_2011), 0) AS population_2011
FROM ward_heat_analysis
WHERE population_2011 IS NOT NULL
GROUP BY heat_level
ORDER BY avg_lst_c DESC;

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT ward_id) AS unique_wards,
    SUM(CASE WHEN population_2011 IS NULL THEN 1 ELSE 0 END) AS missing_population,
    SUM(CASE WHEN population_density_2011 IS NULL THEN 1 ELSE 0 END) AS missing_density,
    SUM(CASE WHEN lst_2025_c IS NULL THEN 1 ELSE 0 END) AS missing_lst,
    SUM(CASE WHEN ndvi_2025 IS NULL THEN 1 ELSE 0 END) AS missing_ndvi,
    SUM(CASE WHEN ndbi_2025 IS NULL THEN 1 ELSE 0 END) AS missing_ndbi
FROM ward_heat_analysis;
"""

with open(sql_analysis_path, "w", encoding="utf-8") as f:
    f.write(sql_content)

print("SQL analysis file created successfully")
print(sql_analysis_path)
print("File size:", os.path.getsize(sql_analysis_path), "bytes")

SQL analysis file created successfully
/content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/sql/analysis_queries.sql
File size: 2704 bytes


In [7]:
with open(sql_analysis_path, "r", encoding="utf-8") as f:
    sql_text = f.read()

print("SQL file verification")
print("---------------------")
print("Characters:", len(sql_text))
print("SELECT statements:", sql_text.upper().count("SELECT"))
print("GROUP BY statements:", sql_text.upper().count("GROUP BY"))
print("File exists:", os.path.exists(sql_analysis_path))

SQL file verification
---------------------
Characters: 2704
SELECT statements: 9
GROUP BY statements: 5
File exists: True


In [8]:
import geopandas as gpd

final_geojson_path = (
    "/content/drive/MyDrive/"
    "Hyderabad_Urban_Heat_Analytics/"
    "data/processed/"
    "hyderabad_urban_heat_analytics_2025.geojson"
)

powerbi_map = gpd.read_file(final_geojson_path)

print("Rows:", len(powerbi_map))
print("Columns:", list(powerbi_map.columns))
print("CRS:", powerbi_map.crs)

Rows: 150
Columns: ['ward_id', 'ward_name', 'circle', 'zone', 'area_sq_km', 'population_2011', 'households_2011', 'population_density_2011', 'lst_2025_c', 'ndvi_2025', 'ndbi_2025', 'heat_percentile', 'vegetation_percentile', 'built_up_percentile', 'density_percentile', 'heat_rank', 'vegetation_rank', 'built_up_intensity_rank', 'heat_level', 'vegetation_level', 'built_up_level', 'heat_density_category', 'geometry']
CRS: EPSG:4326


In [9]:
powerbi_map = powerbi_map[
    ["ward_id", "ward_name", "geometry"]
].copy()

powerbi_map["ward_id"] = powerbi_map["ward_id"].astype(str)

powerbi_geojson_path = (
    "/content/drive/MyDrive/"
    "Hyderabad_Urban_Heat_Analytics/"
    "data/processed/"
    "ghmc_wards_150_powerbi.geojson"
)

powerbi_map.to_file(
    powerbi_geojson_path,
    driver="GeoJSON"
)

print("Power BI GeoJSON created successfully")
print("Rows:", len(powerbi_map))
print("File:", powerbi_geojson_path)

Power BI GeoJSON created successfully
Rows: 150
File: /content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/data/processed/ghmc_wards_150_powerbi.geojson


In [10]:
import json

with open(powerbi_geojson_path, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

print("Features:", len(geojson_data["features"]))

for feature in geojson_data["features"][:5]:
    print(feature["properties"])

Features: 150
{'ward_id': '1', 'ward_name': '1-KAPRA'}
{'ward_id': '2', 'ward_name': '2-DR.A.S.RAO NAGAR'}
{'ward_id': '3', 'ward_name': '3-CHERLAPALLY'}
{'ward_id': '4', 'ward_name': '4-MEERPET H.B.COLONY'}
{'ward_id': '5', 'ward_name': '5-MALLAPUR'}
